# RVC voice model of Talha Zubayer — train on a free Colab GPU (Applio, MIT)

**কীভাবে চালাবেন (৩ ক্লিক):**
1. উপরের মেনু: *Runtime → Change runtime type → T4 GPU* → Save।
2. বাঁ দিকের ফোল্ডার আইকনে ক্লিক করে `dataset.zip` ফাইলটা টেনে ছেড়ে দিন (তৈরি করা আছে: `VideoWork/voice-clone/rvc/dataset.zip`)।
3. *Runtime → Run all*। ৪০–৭০ মিনিট পর শেষ ঘরটা `talha.zip` নামিয়ে দেবে। সেটা `VideoWork/voice-clone/rvc/` ফোল্ডারে রাখুন।

Made 2026-10-30 for the voice-clone project (docs/voice-clone-research.md). Applio: https://github.com/IAHispano/Applio

In [ ]:
# 1. GPU check + Applio install (about 3 min)
import subprocess, os, sys
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True).stdout or 'NO GPU — set Runtime → Change runtime type → T4 GPU')
!git clone -q https://github.com/IAHispano/Applio.git /content/Applio
%cd /content/Applio
!pip install -q -r requirements.txt 2>&1 | tail -2
!python core.py prerequisites --pretraineds-hifigan --models --no-exe 2>&1 | tail -3
print('install done')

In [ ]:
# 2. Dataset: dataset.zip (WAV clips, 48 kHz mono, cleaned) → /content/dataset/talha
import zipfile, glob, os
src = '/content/dataset.zip'
assert os.path.exists(src), 'dataset.zip not found: drag it into the Files pane on the left'
os.makedirs('/content/dataset/talha', exist_ok=True)
with zipfile.ZipFile(src) as z:
    for n in z.namelist():
        if n.lower().endswith('.wav'):
            with open('/content/dataset/talha/' + os.path.basename(n), 'wb') as f: f.write(z.read(n))
wavs = glob.glob('/content/dataset/talha/*.wav')
print(len(wavs), 'clips')
assert len(wavs) >= 20, 'too few clips'

In [ ]:
# 3. Preprocess + extract features (48 kHz, f0 = rmvpe, embedder = contentvec). Flag names checked against core.py --help on 2026-10-31.
%cd /content/Applio
MODEL = 'talha'
!python core.py preprocess --model-name {MODEL} --dataset-path /content/dataset/talha --sample-rate 48000 --cpu-cores 2 --cut-preprocess Automatic --normalization-mode post 2>&1 | tail -3
!python core.py extract --model-name {MODEL} --f0-method rmvpe --sample-rate 48000 --embedder-model contentvec --cpu-cores 2 --gpu 0 2>&1 | tail -3
print('features done')

In [ ]:
# 4. Train (300 epochs on ~17 min of speech ≈ 40–60 min on a T4)
%cd /content/Applio
MODEL = 'talha'
!python core.py train --model-name {MODEL} --sample-rate 48000 --total-epoch 300 --batch-size 8 --save-every-epoch 50 --save-only-latest --save-every-weights --gpu 0 --vocoder HiFi-GAN --pretrained --index-algorithm Auto 2>&1 | tail -5
print('training done')

In [ ]:
# 5. Pack the model (.pth + .index) → talha.zip and download it
import glob, zipfile, os
from google.colab import files
pths = sorted(glob.glob('/content/Applio/logs/talha/talha_*e_*s.pth') + glob.glob('/content/Applio/logs/talha/talha.pth'), key=os.path.getmtime)
idx = glob.glob('/content/Applio/logs/talha/*.index')
assert pths, 'no trained .pth found'
with zipfile.ZipFile('/content/talha.zip', 'w') as z:
    z.write(pths[-1], 'talha.pth')
    if idx: z.write(idx[-1], 'talha.index')
print('packed', os.path.basename(pths[-1]), idx[-1] if idx else '(no index)')
files.download('/content/talha.zip')